# testConditionalGANPytorch1  
Andrew E. Davidson aedavids@ucsc.edu 8/29/24  

Copyright (c) 2020-2023, Regents of the University of California All rights reserved.   https://polyformproject.org/licenses/noncommercial/1.0.0

AIM: create a simple GAN that is easy to test our basic framework

generate y = x^2

ref: 
- chapter 6. in Generative Advisarial Networks with Python  
    this does not work. Keras/tensor flow version issues?  
    re-write example using pytorch

- [pytorch doc](https://pytorch.org/docs/stable/index.html)

In [1]:
import ipynbname
import matplotlib.pyplot as plt

from numpy import hstack
from numpy import zeros
from numpy import ones
from numpy.random import rand
from numpy.random import randn
import os

# by default keras use tensorflow as backend
import torch
print(f'torch.__version__: {torch.__version__}')

from torch import nn
torch.manual_seed(0) # Set for testing purposes, please do not change!

# tqdm provides progress bars for loops and iterables.
from tqdm.auto import tqdm

# class that helps you efficiently load and iterate over your dataset
# during training or inference
# we do not need this for our toy example
from torch.utils.data import DataLoader


notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

outDir = f'{notebookDir}/{notebookName}.out'
imgOut = f'{outDir}/img'
print(f'imgOut:\n{imgOut}')

torch.__version__: 2.5.1.post102
imgOut:
/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/gan/testConditionalGAN_pytorch_1.out/img


/private/home/aedavids/miniconda3/envs/extraCellularRNA/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Create models

In [2]:
class ParabolaGenerator( nn.Module ):
    def __init__( self, latentDim : int, nOutputs : int = 2 ) :
        '''
        arguments:
            latentDim: the length of the noise input vector

            nOutputs : the length of the output layers vector. Default = 2
        '''
        super().__init__()


        # keras
        # model.add(Dense(15, activation='relu', kernel_initializer='he_uniform', input_dim=latent_dim))
        # model.add(Dense(n_outputs, activation='linear'))
        
        self.model = nn.Sequential(
            nn.Linear(latentDim, 15),
            nn.ReLU(),
            nn.Linear(15, nOutputs),
        )
        # Initialize weights
        nn.init.kaiming_uniform_(self.model[0].weight, nonlinearity='relu')
        nn.init.zeros_(self.model[0].bias)
        nn.init.kaiming_uniform_(self.model[2].weight, nonlinearity='linear')
        nn.init.zeros_(self.model[2].bias)

    def forward( self, noise : torch.Tensor ):
        '''
            noise should be the value returned by generateLatentPoints()
        '''
        ret = self.model( noise )
        return ret

In [3]:
class ParabolaDiscriminator( nn.Module ):
    def __init__( self, inputSize : int = 2 ):
        '''
            inputSize, the length of the generated vectors
        '''
        super().__init__()

        # keras
        # model = Sequential()
    	# model.add(Dense(25, activation='relu', kernel_initializer='he_uniform', input_dim=n_inputs))
    	# model.add(Dense(1, activation='sigmoid'))
        
        self.model = nn.Sequential(
            nn.Linear(inputSize, 25),
            nn.ReLU(),
            nn.Linear(25, 1),
            nn.Sigmoid()
        )
        
        # Initialize weights
        nn.init.kaiming_uniform_(self.model[0].weight, nonlinearity='relu')
        nn.init.zeros_(self.model[0].bias)
        nn.init.kaiming_uniform_(self.model[2].weight, nonlinearity='sigmoid')
        nn.init.zeros_(self.model[2].bias)

    def forward( self, X : torch.Tensor ):
        ret = self.model( X )
        return ret

def testParabolaDiscriminator() :
    inputSize = 2
    discriminator = ParabolaDiscriminator( inputSize )
    numTest = 2
    testInput = torch.randn( numTest, inputSize )
    predictions = discriminator( testInput )
    print(f'\npredictions :\n{predictions}')

testParabolaDiscriminator()


predictions :
tensor([[0.5380],
        [0.5828]], grad_fn=<SigmoidBackward0>)


## Data Utilities
function to generate real and fake tensors

In [4]:
def generateRealSamples( n : int ) -> tuple[torch.Tensor, torch.Tensor] :
    '''
    generate n real parabola samples with class labels

    Returns 2 Tensor
        X, y i.e. (realSamples, realLabels)

        y = 1, ie real
    '''
    # generate inputs in range [-0.5, 0.5]
    X1 = rand(n) - 0.5
    
    # generate outputs X^2
    X2 = X1 * X1
    
    # stack arrays
    X1 = X1.reshape(n, 1)
    X2 = X2.reshape(n, 1)
    X = hstack((X1, X2))
    
    # generate class labels
    y = ones((n, 1))
    
    realSamples = torch.Tensor( X )
    realLabels = torch.Tensor( y)
    
    return (realSamples, realLabels)

def testGenerateRealSamples() :
    X, y = generateRealSamples( n = 5 ) 
    print( f'X.shape: {X.shape} rank : {len(X.shape)} num elements : {X.numel()}' )
    print( X )

    print( f'\ny.shape: {y.shape} rank : {len(y.shape)} num elements : {y.numel()}' )
    print ( y) 

testGenerateRealSamples()

X.shape: torch.Size([5, 2]) rank : 2 num elements : 10
tensor([[ 0.4056,  0.1645],
        [-0.3236,  0.1047],
        [ 0.2893,  0.0837],
        [-0.2060,  0.0424],
        [ 0.2162,  0.0468]])

y.shape: torch.Size([5, 1]) rank : 2 num elements : 5
tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.]])


In [5]:
def generateLatentPoints(latentDimensions : int = 5, 
                         n : int = 100) -> torch.Tensor :
    '''
    generate points in latent space as input for the generator

    latentDimensions:
        the number of dimensions for the generator's input vector

    n the number of vectors to generate

    returns a tensor
    '''
    # generate points in the latent space
    xInput = randn(latentDimensions * n)
    
    # reshape into a batch of inputs for the network
    xInput = xInput.reshape(n, latentDimensions)
    
    ret = torch.Tensor( xInput )
    
    return ret

def testGenerateLatentPoints():
    tglp = generateLatentPoints(latentDimensions=5, n=3 )
    print( tglp )
    print( tglp.shape )

testGenerateLatentPoints()

tensor([[ 0.7022,  0.5902,  0.5814,  0.5790,  0.5166],
        [-0.2452,  1.0188, -0.3913, -0.7466, -1.6123],
        [ 0.7608,  0.7354, -0.9204, -1.5086,  0.5178]])
torch.Size([3, 5])


In [6]:
def generateFakeSamples(generator : ParabolaGenerator, 
                        latentDimensions : int,
                        n : int) -> tuple[torch.Tensor, torch.Tensor]:
    '''
    use the generator to generate n fake examples, with class labels

    returns (fakeSamples, fakeLabels)
        labels will be zeros.
    '''
    # generate points in latent space
    noiseVector = generateLatentPoints(latentDimensions, n)

    # Set the model to evaluation mode
    # Layers like Dropout and Batch Normalization behave differently during 
    # training and evaluation.
    #
    # we do not need to worry about this. It a good future proofing code example
    generator.eval()

    # Disable gradient calculation for inference
    with torch.no_grad():
        fakeSamples = generator(noiseVector)
    
    # create class labels  
    y = zeros((n, 1))
    fakeLabels = torch.Tensor( y )
    
    return fakeSamples, fakeLabels

def testGenerateFakeSamples():
    latentDimensions = 5
    numOutputs = 2
    

    gen = ParabolaGenerator( latentDimensions, numOutputs )

    numSamples = 3
    fakeSamples, fakeLabels = generateFakeSamples( gen, latentDimensions, numSamples)

    print( f'\nfakeSamples.shape: {fakeSamples.shape} rank : {len(fakeSamples.shape)} num elements : {fakeSamples.numel()}' )
    print( fakeSamples )

    print( f'\nfakeLabels.shape: {fakeLabels.shape} rank : {len(fakeLabels.shape)} num elements : {fakeLabels.numel()}' )
    print ( fakeLabels ) 


testGenerateFakeSamples()


fakeSamples.shape: torch.Size([3, 2]) rank : 2 num elements : 6
tensor([[ 3.2058, -1.2371],
        [ 1.1496,  1.2469],
        [ 2.2227, -1.1514]])

fakeLabels.shape: torch.Size([3, 1]) rank : 2 num elements : 3
tensor([[0.],
        [0.],
        [0.]])


## Train Model

In [23]:
def getDiscriminatorLoss(generator, discriminator, criterion, realSamples, n, zDim, device):
    '''
    Return the loss of the discriminator given inputs.
    Parameters:
        generator: the generator model, which returns an image given z-dimensional noise
        
        discriminator: the discriminator model, which returns a single-dimensional prediction of real/fake
        
        criterion: the loss function, which should be used to compare 
               the discriminator's predictions to the ground truth reality of the images 
               (e.g. fake = 0, real = 1)
               
        real: a batch of real samples
        
        n: the number of images the generator should produce, 
                which is also the length of the real images
                
        zDim: the dimension of the noise vector, a scalar
        
        device: the device type
        
    Returns:
        discriminatorLoss: a torch scalar loss value for the current batch
    '''
    # generate some fake samples
    fakeSamples, fakeLabels  = generateFakeSamples( generator, zDim, n )

    # get discriminator's prediction of the fake samples
    predictionsOnFakes = discriminator( fakeSamples )
    
    # calculate the loss on fakes
    expected = torch.zeros(n, 1, device=device)  # All weights are equal to 0, ie fake
   
    # https://pytorch.org/docs/master/generated/torch.nn.BCEWithLogitsLoss.html?highlight=bcewithlogitsloss#torch.nn.BCEWithLogitsLoss
    #discriminatorLossOnFake = criterion(weight=predictionsOnFake, pos_weight=pos_weight) 
    discriminatorLossOnFake = criterion(predictionsOnFakes, expected) 
    #print(f'\n discriminatorLossOnFake.shape:{discriminatorLossOnFake.shape}')
    #print(f'discriminatorLossOnFake:\n{discriminatorLossOnFake}')
    
    # calculate loss on real samples
    predictionsOnReal = discriminator( realSamples )
    print(f'realSamples.shape : {realSamples.shape}')
    
    #pos_weight = torch.ones([num_images])  # All weights are equal to 1, ie real
    expected = torch.ones(n, 1, device=device)  # All weights are equal to 1, ie real
    print(f'predictionsOnReal.shape : {predictionsOnReal.shape}')
    print(f'expected.shape : {expected.shape}')    
    discriminatorLossOnReal = criterion(predictionsOnReal, expected) 
    #print(f'\ndiscriminatorLossOnReal.shape:{discriminatorLossOnReal.shape}')
    #print(f'discriminatorLossOnReal:\n{discriminatorLossOnReal}')
    
    # calculate the average loss
    discriminatorLoss = (discriminatorLossOnReal + discriminatorLossOnFake) / 2
    
    return discriminatorLoss

In [24]:
def testGetDiscriminatorLossSample() :
    criterion = nn.BCEWithLogitsLoss()
    nEpochs = 2 # 200
    numBatchs = 3 # number of batchs per epoch
    zDim = 5 # length of noise vector
    displayStep = 1 # 500
    batchSize = 4
    lr = 0.00001
    exampleLength = 2 # X,Y
    #device = 'cuda'
    device = 'cpu'
    generator= ParabolaGenerator(latentDim=zDim , nOutputs=exampleLength).to(device)
    #generatorOptimizer = torch.optim.Adam(generator.parameters(), lr=lr)
    
    discriminator = ParabolaDiscriminator(inputSize=exampleLength).to(device) 
    #discriminatorOptimizer = torch.optim.Adam(discriminator.parameters(), lr=lr)

    realSamples, realLabels = generateRealSamples( batchSize )
    aedwip = getDiscriminatorLoss(generator, discriminator, criterion, realSamples, batchSize,
                                  zDim, device)
    print( type(aedwip))
    print( aedwip )

testGetDiscriminatorLossSample()

realSamples.shape : torch.Size([4, 2])
predictionsOnReal.shape : torch.Size([4, 1])
expected.shape : torch.Size([4, 1])
<class 'torch.Tensor'>
tensor(0.6250, grad_fn=<DivBackward0>)


In [ ]:
aedwip

In [ ]:
# Set your parameters
criterion = nn.BCEWithLogitsLoss()
nEpochs = 2 # 200
numBatchs = 3 # number of batchs per epoch
zDim = 5 # length of noise vector
displayStep = 1 # 500
batchSize = 128
lr = 0.00001
exampleLength = 2 # X,Y
#device = 'cuda'
device = 'cpu'

generator= ParabolaGenerator(latentDim=zDim , nOutputs=exampleLength).to(device)
generatorOptimizer = torch.optim.Adam(generator.parameters(), lr=lr)

discriminator = ParabolaDiscriminator(inputSize=exampleLength).to(device) 
discriminatorOptimizer = torch.optim.Adam(discriminator.parameters(), lr=lr)

In [ ]:
currentStep = 0
meanGeneratorLoss = 0
meanDiscriminatorLoss = 0
testGenerator = True # Whether the generator should be tested
genLoss = False
error = False


for epoch in range(nEpochs):
    print(f'epoch: {epoch}')    
    #  provides progress bars for loops and iterables.
    # https://tqdm.github.io/
    for i in tqdm( range(numBatchs) ) : 
        # if we where working with real data it is possible 
        # some batches are short. are < batchSize
        currentBatchSize = batchSize
        realSamples = generateRealSamples( currentBatchSize )
        
        ### Update discriminator ###
        # Zero out the gradients before backpropagation
        discriminatorOptimizer.zero_grad()
        
        # Calculate discriminator loss
        discLoss = get_disc_loss(generator, discriminator, criterion, realSamples, 
                                  currentBatchSize, zDim, device)
        
        # Update gradients
        discLoss.backward(retain_graph=True)
        
        # Update optimizer
        disc_opt.step()


In [ ]:
this is junk

# create a discrimanator
discriminatorModel = ParabolaDiscriminator( inputSize=2 )
# define the cost function
discriminatorCriterion = nn.BCELoss()

# use stochastic gradient decent
# keras
# 	discrModel.compile(loss='binary_crossentropy', optimizer='adam', 
# metrics=['accuracy'])

learningRate = 0.01
discriminatorOptimizer = torch.optim.adam( discriminatorModel.parameters, 
                                          lr=learningRate )

# define the number of training loops
numEpochs = aedwip
for t in range( numEpochs ) :

    # forward propagation
    # get a prediction
    yHat = discriminatorModel( X )
    discriminatorLoss = discriminatorCriterion( yHat, y)

    # backward propagation
    discriminatorOptimizer.zero_grad()
    discriminatorLoss.backward()

    # update the parameters
    discriminatorOptimizer.step()
    